# 05b - (Tahap 3) Evaluasi dataset QA

In [1]:
!pip install -q sentence-transformers nltk

In [2]:
import json, os, csv, math, random, datetime
from collections import defaultdict, Counter
from pathlib import Path
 
import torch
import numpy as np
import nltk
nltk.download("punkt", quiet=True)
 
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device={DEVICE}")

device=cuda


## 1. Konfigurasi

In [3]:
CONFIG = {
    "input_path":  "/kaggle/input/notebooks/muhammadnabilul/05b-tahap-2-cleaning-dataset-qa/output_05b/filtered_records.jsonl",
    "removed_path": "/kaggle/input/notebooks/muhammadnabilul/05b-tahap-2-cleaning-dataset-qa/output_05b/removed_records.jsonl",
    "output_dir": "output_05b_tahap3",
 
    "diversity_embed_model": "firqaaa/Indo-Sentence-BERT-Base",
    "diversity_embed_batch": 32,
    "opener_ngram_n": 3,
    "opener_top_k":   15,
 
    "stratify_key": "subtask_type",
 
    "precision_check_n_per_flag": 20,
    "precision_check_n_lolos":    40,
    "precision_check_seed": 7,
 
    "seed": 42,
}
 
Path(CONFIG["output_dir"]).mkdir(parents=True, exist_ok=True)
random.seed(CONFIG["seed"])
os.environ["PYTHONHASHSEED"] = str(CONFIG["seed"])

## Fungsi Umum

In [4]:
def load_records(path):
    """Muat seluruh record dari file JSONL menjadi list of dict."""
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]
 
def write_csv(rows, path, fieldnames):
    """Tulis list of dict sebagai CSV dengan header eksplisit."""
    with open(path, "w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        for r in rows:
            w.writerow(r)
 
def write_json(obj, path):
    """Tulis dict/list sebagai file JSON dengan indentasi rapi."""
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
 
records = load_records(CONFIG["input_path"])
print(f"Total filtered records: {len(records)}")

Total filtered records: 9350


## 2. Diversity Metric

In [5]:
def tokenize(text):
    """Tokenisasi sederhana berbasis whitespace, lowercase."""
    return text.lower().split()
 
def ngrams(tokens, n):
    """Hasilkan seluruh n-gram dari daftar token."""
    return [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]
 
def distinct_n(questions, n):
    """
    Hitung rasio n-gram unik terhadap total n-gram pada seluruh
    instruction -- indikator keberagaman leksikal, dihitung dari
    POPULASI PENUH (bukan sample).
    """
    total, unique = 0, set()
    for q in questions:
        g = ngrams(tokenize(q), n)
        total += len(g)
        unique.update(g)
    return len(unique) / total if total else 0.0
 
def self_bleu(questions, seed, max_len_words=40):
    """
    Ukur kemiripan rata-rata tiap instruction terhadap SEMUA instruction
    lain (self-BLEU) pada POPULASI PENUH -- makin rendah, makin beragam.
    Kompleksitas O(n^2) sehingga untuk populasi besar (>5000) proses ini
    memakan waktu signifikan (puluhan menit hingga >1 jam); dijalankan
    tanpa sampling sesuai keputusan mencakup seluruh populasi. Panjang
    tiap kalimat dibatasi max_len_words untuk kestabilan runtime tanpa
    mengubah esensi pengukuran (BLEU pada kalimat sangat panjang tidak
    lebih informatif, hanya lebih lambat dihitung).
    """
    from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
    toks = [tokenize(q)[:max_len_words] for q in questions]
    sm = SmoothingFunction().method1
    scores = []
    for i, hyp in enumerate(toks):
        refs = toks[:i] + toks[i+1:]
        if refs and hyp:
            scores.append(sentence_bleu(refs, hyp, smoothing_function=sm))
    return sum(scores) / len(scores) if scores else 0.0
 
def embedding_diversity(questions, model_name, seed, device, batch_size):
    """
    Hitung rata-rata cosine similarity berpasangan antar SELURUH
    instruction (populasi penuh) -- proxy keberagaman semantik.
    Menggunakan matrix multiplication ter-vektorisasi (GPU jika
    tersedia) sehingga tetap efisien untuk populasi berukuran ribuan.
    """
    from sentence_transformers import SentenceTransformer, util
    model = SentenceTransformer(model_name, device=device)
    emb = model.encode(questions, convert_to_tensor=True, batch_size=batch_size,
                       normalize_embeddings=True, show_progress_bar=True)
    sim = util.cos_sim(emb, emb)
    n = sim.shape[0]
    return (sim.sum().item() - n) / (n * (n - 1)) if n > 1 else 0.0
 
def gini_coefficient(counts):
    """Hitung koefisien Gini dari distribusi hitungan kategori (0=merata sempurna, mendekati 1=timpang)."""
    values = sorted(counts)
    n = len(values)
    if n == 0 or sum(values) == 0:
        return 0.0
    weighted_sum = sum((i+1) * v for i, v in enumerate(values))
    total = sum(values)
    return (2 * weighted_sum) / (n * total) - (n + 1) / n
 
def stratum_coverage(records, stratify_key):
    """Hitung distribusi record per kategori, normalized entropy, dan Gini coefficient -- seluruhnya dari populasi penuh."""
    by = defaultdict(int)
    for r in records:
        by[r.get(stratify_key, "?")] += 1
    total = sum(by.values())
    k = len(by)
    props = [v / total for v in by.values()]
    entropy = -sum(p * math.log(p) for p in props if p > 0)
    norm_entropy = entropy / math.log(k) if k > 1 else 0.0
    gini = gini_coefficient(list(by.values()))
    return dict(by), round(norm_entropy, 4), round(gini, 4)
 
def opener_distribution(questions, n_words, top_k):
    """Hitung distribusi frasa pembuka (n kata pertama) instruction -- mendeteksi dominasi pola kalimat tertentu."""
    openers = Counter()
    for q in questions:
        words = tokenize(q)
        opener = " ".join(words[:n_words])
        if opener:
            openers[opener] += 1
    total = sum(openers.values())
    rows = [
        {"opener": o, "count": c, "proportion": round(c/total, 4)}
        for o, c in openers.most_common(top_k)
    ]
    return rows, total

In [6]:
def diversity_report(records, config):
    """Rakit seluruh metrik diversity dari POPULASI PENUH, simpan ke JSON+CSV, cetak ke layar."""
    questions = [r.get("instruction", "") for r in records if r.get("instruction", "").strip()]
    n_q = len(questions)
 
    d1 = distinct_n(questions, 1)
    d2 = distinct_n(questions, 2)
 
    n_pairs_est = n_q * (n_q - 1)
    print(f"\n[PERINGATAN] self_bleu akan memproses ~{n_pairs_est:,} pasangan kalimat "
          f"(populasi penuh N={n_q}). Ini dapat memakan waktu signifikan.")
    print("menghitung self_bleu (populasi penuh, mohon tunggu) ...")
    sb = self_bleu(questions, config["seed"])
 
    print("menghitung mean_pairwise_cosine (populasi penuh, GPU jika tersedia) ...")
    mpc = embedding_diversity(questions, config["diversity_embed_model"], config["seed"], DEVICE, config["diversity_embed_batch"])
 
    stratum_counts, norm_entropy, gini = stratum_coverage(records, config["stratify_key"])
    opener_rows, opener_total = opener_distribution(questions, config["opener_ngram_n"], config["opener_top_k"])
    top5_share = sum(r["proportion"] for r in opener_rows[:5]) if opener_rows else 0.0
 
    summary = {
        "n_questions_total_population": n_q,
        "distinct_1": round(d1, 4),
        "distinct_2": round(d2, 4),
        "self_bleu": round(sb, 4),
        "mean_pairwise_cosine": round(mpc, 4),
        "stratum_counts": stratum_counts,
        "stratum_normalized_entropy": norm_entropy,
        "stratum_gini": gini,
        "top5_opener_share": round(top5_share, 4),
        "interpretation": (
            "Seluruh metrik dihitung dari POPULASI PENUH (N={}), tanpa sampling. "
            "distinct_n, normalized_entropy: makin tinggi makin beragam/merata. "
            "self_bleu, mean_pairwise_cosine, gini: makin RENDAH makin beragam/merata."
        ).format(n_q),
    }
    write_json(summary, f"{config['output_dir']}/diversity_summary.json")
    write_csv(opener_rows, f"{config['output_dir']}/opener_distribution.csv", ["opener", "count", "proportion"])
 
    print("\n--- diversity summary (populasi penuh) ---")
    for k, v in summary.items():
        if k not in ("stratum_counts", "interpretation"):
            print(f"  {k}: {v}")
    print("\n  stratum_counts:")
    for st, c in sorted(stratum_counts.items()):
        print(f"    {st:<32}{c:>6}")
    print(f"\n  konsentrasi pembuka ({config['opener_ngram_n']} kata pertama), top-5:")
    for r in opener_rows[:5]:
        print(f"    {r['count']:>6,} ({r['proportion']*100:>5.2f}%)  {r['opener']}")
 
    return summary
 
diversity_summary = diversity_report(records, CONFIG)


[PERINGATAN] self_bleu akan memproses ~87,413,150 pasangan kalimat (populasi penuh N=9350). Ini dapat memakan waktu signifikan.
menghitung self_bleu (populasi penuh, mohon tunggu) ...
menghitung mean_pairwise_cosine (populasi penuh, GPU jika tersedia) ...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/118 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/293 [00:00<?, ?it/s]


--- diversity summary (populasi penuh) ---
  n_questions_total_population: 9350
  distinct_1: 0.0515
  distinct_2: 0.2752
  self_bleu: 0.6192
  mean_pairwise_cosine: 0.282
  stratum_normalized_entropy: 0.9845
  stratum_gini: 0.1357
  top5_opener_share: 0.1971

  stratum_counts:
    causal_reasoning                   934
    comparative_analysis               942
    definition_clarification          1344
    document_overview                  515
    factual_retrieval                 1270
    methodology_explanation           1296
    multi_source_synthesis             850
    source_navigation                 1279
    trend_interpretation               920

  konsentrasi pembuka (3 kata pertama), top-5:
       946 (10.12%)  apa yang dimaksud
       358 ( 3.83%)  di bagian mana
       217 ( 2.32%)  pada tabel nomor
       211 ( 2.26%)  uraikan gambaran menyeluruh
       110 ( 1.18%)  di tabel nomor


## 3. Indobert Similarity (Jawaban + Konteks)

In [7]:
def compute_indobert_similarity(records, model_name, device, batch_size):
    """
    Hitung cosine similarity embedding antara response dan
    context_joined untuk SETIAP record (populasi penuh). Similarity
    rendah mengindikasikan response yang secara semantik menyimpang
    dari konteks -- kandidat prioritas untuk precision-check manual.
    """
    from sentence_transformers import SentenceTransformer, util
    model = SentenceTransformer(model_name, device=device)
    answers = [r["response"] for r in records]
    contexts = [
        r.get("context_joined") or "\n\n---\n\n".join(r.get("context", []))
        for r in records
    ]
    emb_a = model.encode(answers, convert_to_tensor=True, batch_size=batch_size, show_progress_bar=True)
    emb_c = model.encode(contexts, convert_to_tensor=True, batch_size=batch_size, show_progress_bar=True)
    scores = util.cos_sim(emb_a, emb_c).diagonal().tolist()
    return scores

In [8]:
def indobert_report(records, config):
    """Hitung similarity untuk seluruh populasi, ringkas per stratum, simpan CSV lengkap + JSON ringkasan."""
    scores = compute_indobert_similarity(
        records, config["diversity_embed_model"], DEVICE, config["diversity_embed_batch"]
    )
    rows = []
    for r, s in zip(records, scores):
        rows.append({
            "record_id": r["record_id"],
            "subtask_type": r.get(config["stratify_key"], "?"),
            "indobert_score": round(s, 4),
        })
    write_csv(rows, f"{config['output_dir']}/indobert_similarity_full.csv",
              ["record_id", "subtask_type", "indobert_score"])
 
    by_stratum = defaultdict(list)
    for r in rows:
        by_stratum[r["subtask_type"]].append(r["indobert_score"])
 
    overall_scores = [r["indobert_score"] for r in rows]
    overall_mean = float(np.mean(overall_scores))
    overall_median = float(np.median(overall_scores))
 
    print(f"\n{'Subtask':<32}{'N':>6}{'Mean':>9}{'Median':>9}{'Min':>9}")
    stratum_stats = {}
    for st, vals in sorted(by_stratum.items()):
        mean_v, median_v, min_v = float(np.mean(vals)), float(np.median(vals)), float(np.min(vals))
        stratum_stats[st] = {"n": len(vals), "mean": round(mean_v, 4), "median": round(median_v, 4), "min": round(min_v, 4)}
        print(f"{st:<32}{len(vals):>6}{mean_v:>9.3f}{median_v:>9.3f}{min_v:>9.3f}")
 
    n_low = sum(1 for s in overall_scores if s < 0.5)
    summary = {
        "n_records_total_population": len(rows),
        "mean_overall": round(overall_mean, 4),
        "median_overall": round(overall_median, 4),
        "pct_below_0.5": round(n_low / len(rows) * 100, 2),
        "by_stratum": stratum_stats,
        "interpretation": (
            "Dihitung untuk SELURUH populasi filtered_records (N={}), bukan sample -- model "
            "IndoBERT lokal, tidak memerlukan API berbayar sehingga tidak ada batasan biaya "
            "yang memaksa sampling. Similarity rendah (<0.5) menandakan response yang secara "
            "semantik kurang selaras dengan context_joined, dijadikan kandidat prioritas pada "
            "precision-check manual (lihat precision_check_removed.csv/lolos.csv)."
        ).format(len(rows)),
    }
    write_json(summary, f"{config['output_dir']}/indobert_similarity_summary.json")
 
    print(f"\nMean similarity keseluruhan   : {overall_mean:.4f}")
    print(f"Median similarity keseluruhan : {overall_median:.4f}")
    print(f"Persentase di bawah 0.5       : {n_low}/{len(rows)} ({n_low/len(rows)*100:.2f}%)")
 
    return summary, rows
 
indobert_summary, indobert_rows = indobert_report(records, CONFIG)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: firqaaa/Indo-Sentence-BERT-Base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/293 [00:00<?, ?it/s]

Batches:   0%|          | 0/293 [00:00<?, ?it/s]


Subtask                              N     Mean   Median      Min
causal_reasoning                   934    0.818    0.872    0.175
comparative_analysis               942    0.835    0.879    0.189
definition_clarification          1344    0.831    0.868    0.026
document_overview                  515    0.841    0.884    0.284
factual_retrieval                 1270    0.852    0.907    0.175
methodology_explanation           1296    0.816    0.846    0.189
multi_source_synthesis             850    0.839    0.877    0.303
source_navigation                 1279    0.708    0.747    0.048
trend_interpretation               920    0.833    0.889    0.202

Mean similarity keseluruhan   : 0.8155
Median similarity keseluruhan : 0.8635
Persentase di bawah 0.5       : 600/9350 (6.42%)


## 4. Ekspor CSV

In [9]:
def export_filtered_records_csv(records, indobert_rows, path):
    """Gabungkan filtered_records dengan indobert_score, ekspor sebagai CSV siap-baca manual."""
    score_map = {r["record_id"]: r["indobert_score"] for r in indobert_rows}
    fieldnames = [
        "record_id", "task_type", "subtask_type", "instruction", "cot",
        "response", "context_joined", "n_context_spans", "document_title",
        "document_year", "section_id", "section_title",
        "track_a_score", "numeric_score", "indobert_score",
    ]
    with open(path, "w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        w.writeheader()
        for r in records:
            row = dict(r)
            row["context_joined"] = row.get("context_joined") or " ||| ".join(row.get("context", []))
            row["n_context_spans"] = len(row.get("context", []))
            row["indobert_score"] = score_map.get(r["record_id"], "")
            w.writerow(row)
 
export_filtered_records_csv(records, indobert_rows, f"{CONFIG['output_dir']}/filtered_records_full.csv")
print(f"\nCSV gabungan filtered_records + skor evaluasi disimpan ke: "
      f"{CONFIG['output_dir']}/filtered_records_full.csv")


CSV gabungan filtered_records + skor evaluasi disimpan ke: output_05b_tahap3/filtered_records_full.csv


## 5. Precision-check

### 5a. sample terbuang
mengecek False Positive: apakah keputusan buang benar

In [10]:
removed_records = load_records(CONFIG["removed_path"])
print(f"Total removed records: {len(removed_records)}")
 
HARD_GATE_KEYS = [
    "high_ocr_noise", "mixed_english_detected", "schema_invalid",
    "chunk_id_invalid", "degenerate", "table_ref_instruction",
    "near_duplicate", "table_residue",
]
 
def sample_removed_per_flag(removed_records, flag_keys, n_per_flag, seed):
    """Ambil sample acak record yang dibuang, dipisah per alasan (flag), untuk pemeriksaan manual False Positive."""
    rng = random.Random(seed)
    rows = []
    for flag in flag_keys:
        candidates = [r for r in removed_records if flag in r.get("flags", {})]
        sample = rng.sample(candidates, min(n_per_flag, len(candidates)))
        for r in sample:
            rows.append({
                "flag_diperiksa": flag,
                "record_id": r["record_id"],
                "subtask_type": r.get("subtask_type", ""),
                "instruction": r.get("instruction", ""),
                "response": r.get("response", ""),
                "cot": r.get("cot", ""),
                "context_joined": (r.get("context_joined", "") or "")[:500],
                "alasan_dibuang": str(r["flags"].get(flag, "")),
                "keputusan_manual": "",
                "catatan": "",
            })
        print(f"  flag={flag:<28} kandidat={len(candidates):>6} sample={len(sample)}")
    return rows
 
removed_sample_rows = sample_removed_per_flag(
    removed_records, HARD_GATE_KEYS,
    CONFIG["precision_check_n_per_flag"], CONFIG["precision_check_seed"],
)
 
removed_fieldnames = [
    "flag_diperiksa", "record_id", "subtask_type", "instruction", "response",
    "cot", "context_joined", "alasan_dibuang", "keputusan_manual", "catatan",
]
write_csv(removed_sample_rows, f"{CONFIG['output_dir']}/precision_check_removed.csv", removed_fieldnames)
print(f"\nTotal sample precision-check (removed): {len(removed_sample_rows)}")
print(f"Disimpan ke: {CONFIG['output_dir']}/precision_check_removed.csv")

Total removed records: 2911
  flag=high_ocr_noise               kandidat=   749 sample=20
  flag=mixed_english_detected       kandidat=    80 sample=20
  flag=schema_invalid               kandidat=     0 sample=0
  flag=chunk_id_invalid             kandidat=     0 sample=0
  flag=degenerate                   kandidat=     0 sample=0
  flag=table_ref_instruction        kandidat=    40 sample=20
  flag=near_duplicate               kandidat=   232 sample=20
  flag=table_residue                kandidat=  1896 sample=20

Total sample precision-check (removed): 100
Disimpan ke: output_05b_tahap3/precision_check_removed.csv


### 5b. Sample record lolos

mengecek False Negative. DIPERKUAT: separuh kuota diambil dari record ber-indobert_score TERENDAH (kandidat paling mencurigakan menurut similarity semantik)

In [11]:
def sample_lolos_prioritized(records, indobert_rows, n, seed):
    """
    Ambil sample record lolos filter untuk spot-check False Negative:
    separuh dari record ber-similarity terendah (paling mencurigakan),
    separuh acak murni dari sisanya.
    """
    rng = random.Random(seed)
    score_map = {r["record_id"]: r["indobert_score"] for r in indobert_rows}
 
    sorted_by_score = sorted(records, key=lambda r: score_map.get(r["record_id"], 1.0))
    n_low = n // 2
    low_score_candidates = sorted_by_score[:max(n_low * 3, n_low)]
    picked_low = rng.sample(low_score_candidates, min(n_low, len(low_score_candidates)))
    picked_low_ids = {r["record_id"] for r in picked_low}
 
    remaining = [r for r in records if r["record_id"] not in picked_low_ids]
    picked_random = rng.sample(remaining, min(n - len(picked_low), len(remaining)))
 
    rows = []
    for r, source in [(x, "low_similarity") for x in picked_low] + [(x, "random") for x in picked_random]:
        rows.append({
            "record_id": r["record_id"],
            "subtask_type": r.get("subtask_type", ""),
            "sumber_sample": source,
            "indobert_score": score_map.get(r["record_id"], ""),
            "instruction": r.get("instruction", ""),
            "response": r.get("response", ""),
            "cot": r.get("cot", ""),
            "context_joined": (r.get("context_joined", "") or "")[:500],
            "track_a_score": r.get("track_a_score", ""),
            "ada_masalah_tersembunyi": "",
            "catatan": "",
        })
    return rows
 
lolos_sample_rows = sample_lolos_prioritized(
    records, indobert_rows, CONFIG["precision_check_n_lolos"], CONFIG["precision_check_seed"] + 1,
)
 
lolos_fieldnames = [
    "record_id", "subtask_type", "sumber_sample", "indobert_score", "instruction",
    "response", "cot", "context_joined", "track_a_score",
    "ada_masalah_tersembunyi", "catatan",
]
write_csv(lolos_sample_rows, f"{CONFIG['output_dir']}/precision_check_lolos.csv", lolos_fieldnames)
print(f"Total sample precision-check (lolos): {len(lolos_sample_rows)} "
      f"({sum(1 for r in lolos_sample_rows if r['sumber_sample']=='low_similarity')} dari low_similarity, "
      f"{sum(1 for r in lolos_sample_rows if r['sumber_sample']=='random')} acak)")
print(f"Disimpan ke: {CONFIG['output_dir']}/precision_check_lolos.csv")

Total sample precision-check (lolos): 40 (20 dari low_similarity, 20 acak)
Disimpan ke: output_05b_tahap3/precision_check_lolos.csv


## Final Report

In [12]:
final_report = {
    "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "total_filtered_records": len(records),
    "total_removed_records": len(removed_records),
    "diversity": diversity_summary,
    "indobert_similarity": indobert_summary,
    "precision_check": {
        "n_sample_removed": len(removed_sample_rows),
        "n_sample_lolos": len(lolos_sample_rows),
        "status": "menunggu pengisian manual pada precision_check_removed.csv dan precision_check_lolos.csv",
        "file_removed": "precision_check_removed.csv",
        "file_lolos": "precision_check_lolos.csv",
    },
}
write_json(final_report, f"{CONFIG['output_dir']}/tahap3_report.json")
 
print("\n" + "="*60)
print("RINGKASAN TAHAP 3 UNTUK BAB 4")
print("="*60)
print(f"Total record final (setelah Tahap 2)   : {len(records):,}")
print(f"Distinct-1 / Distinct-2                : {diversity_summary['distinct_1']:.3f} / {diversity_summary['distinct_2']:.3f}")
print(f"Self-BLEU                              : {diversity_summary['self_bleu']:.3f}")
print(f"Normalized entropy (stratum)           : {diversity_summary['stratum_normalized_entropy']:.3f}")
print(f"Gini coefficient (stratum)             : {diversity_summary['stratum_gini']:.3f}")
print(f"IndoBERT similarity mean (populasi penuh): {indobert_summary['mean_overall']:.3f}")
print(f"IndoBERT similarity < 0.5               : {indobert_summary['pct_below_0.5']:.2f}%")
print(f"\nSample precision-check disiapkan:")
print(f"  removed : {len(removed_sample_rows)} record (isi manual kolom 'keputusan_manual')")
print(f"  lolos   : {len(lolos_sample_rows)} record, separuh diprioritaskan dari similarity terendah "
      f"(isi manual kolom 'ada_masalah_tersembunyi')")
print(f"\nSetelah diisi manual, hitung precision per flag dengan:")
print(f"  precision = jumlah('BENAR_DIBUANG') / total_sample_flag_tsb")
print(f"\nOutput: {CONFIG['output_dir']}/")


RINGKASAN TAHAP 3 UNTUK BAB 4
Total record final (setelah Tahap 2)   : 9,350
Distinct-1 / Distinct-2                : 0.051 / 0.275
Self-BLEU                              : 0.619
Normalized entropy (stratum)           : 0.985
Gini coefficient (stratum)             : 0.136
IndoBERT similarity mean (populasi penuh): 0.816
IndoBERT similarity < 0.5               : 6.42%

Sample precision-check disiapkan:
  removed : 100 record (isi manual kolom 'keputusan_manual')
  lolos   : 40 record, separuh diprioritaskan dari similarity terendah (isi manual kolom 'ada_masalah_tersembunyi')

Setelah diisi manual, hitung precision per flag dengan:
  precision = jumlah('BENAR_DIBUANG') / total_sample_flag_tsb

Output: output_05b_tahap3/
